# Part 1 – Data extraction and understanding

This notebook connects to the original SQLite database, extracts the raw bus stop-level data, performs basic cleaning and consistency checks, and exports a single canonical CSV that will be reused in all subsequent notebooks.

> **Note:** this notebook reads from the private, cleaned SQLite database (`transport_data_clean.db`). That file is not distributed with this repo, since it contains the full ~7.4 million-row raw collection. If you don't have a copy of it, you won't be able to run this notebook end to end. Use [`data/sample/`](../data/sample/) instead to follow the rest of the pipeline (notebooks 02-04 and Slides) on a small, reproducible sample.

## Load raw data from the source database

In this section we connect to the SQLite database provided for the project and load the raw tables that contain stop-level observations for public transport in Luxembourg.

In [1]:
import sqlite3
import pandas as pd
import os

# Path to the private, cleaned SQLite database. Not distributed with this repo;
# override with your own local path via the TRANSPORT_CLEAN_DB_PATH env var.
DB_PATH = os.environ.get("TRANSPORT_CLEAN_DB_PATH", "../data/transport_data_clean.db")

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

# Open a connection to the SQLite database.
# I will reuse this connection for several SQL queries in this notebook.
conn = sqlite3.connect(DB_PATH)

In [2]:
# Inspect the schema of the main table used in this project.
schema = pd.read_sql_query("PRAGMA table_info('dl_copia');", conn)
schema

,cid,name,type,notnull,dflt_value,pk
0,0,stop_id,TEXT,0,None,0
1,1,trip_id,TEXT,0,None,0
2,2,planned_time,TEXT,0,None,0
3,3,real_time,TEXT,0,None,0
4,4,delay_minutes,REAL,0,None,0
5,5,collected_at,TEXT,0,None,0
6,6,operator,TEXT,0,None,0
7,7,transport_type,TEXT,0,None,0
8,8,planned_seconds,INTEGER,0,None,0
9,9,route_id,INTEGER,0,None,0


In [3]:
BASE_QUERY = """
SELECT
    stop_id,
    trip_id,
    planned_time,
    real_time,
    delay_minutes,
    collected_at,
    operator,
    transport_type,
    route_id,
    route_short_name
FROM dl_copia
"""
df = pd.read_sql_query(BASE_QUERY, conn)
print("Rows loaded:", len(df))
df.head()


Rows loaded: 7356700


,stop_id,trip_id,planned_time,real_time,delay_minutes,collected_at,operator,transport_type,route_id,route_short_name
0,201103001,1|4421|1|82|25072025,2025-07-25 20:12:00,2025-07-25 20:13:00,1.0,2025-07-25 20:13:32,Régime Général des Transports Routiers,Bus,3209,413
1,201103001,1|7769|0|82|25072025,2025-07-25 20:18:00,2025-07-25 20:18:00,0.0,2025-07-25 20:13:32,Régime Général des Transports Routiers,Bus,3208,412
2,201103001,1|4404|4|82|25072025,2025-07-25 20:26:00,2025-07-25 20:26:00,0.0,2025-07-25 20:13:32,Régime Général des Transports Routiers,Bus,3208,412
3,201103001,1|7420|0|82|25072025,2025-07-25 20:27:00,2025-07-25 20:27:00,0.0,2025-07-25 20:13:32,Régime Général des Transports Routiers,Bus,3208,412
4,201103001,1|4443|10|82|25072025,2025-07-25 20:33:00,2025-07-25 20:33:00,0.0,2025-07-25 20:13:32,Régime Général des Transports Routiers,Bus,3210,501


In [4]:
# Load all filtered rows (no sampling)
df = pd.read_sql_query(BASE_QUERY, conn)

print("Data loaded into df.")
print("df.shape:", df.shape)

Data loaded into df.
df.shape: (7356700, 10)


## Export canonical dataset for the project

Finally, we select the ten core columns required by the project and write them to a CSV file.  
This file is the **single source of truth** for Parts 2 (EDA) and 3 (modeling).

In [5]:
import os

# Create a 'data' folder one level above the notebooks directory, if it does not exist yet.
project_data_dir = "../data"
os.makedirs(project_data_dir, exist_ok=True)

# Define a clear file name for the sampled dataset.
csv_path = os.path.join(project_data_dir, "bus_delays_sample.csv")

# Save the sampled DataFrame to CSV so I can reuse it in other notebooks.
df.to_csv(csv_path, index=False)

print("Saved sampled data to:", csv_path)

Saved sampled data to: ../data/bus_delays_sample.csv
